# PaperScraper local vLLM workflow

This notebook shows a complete local workflow using bash cells. It assumes you are already running the notebook in the correct Python environment, with PaperScraper, vLLM, CUDA, and any API credentials already available.

## 1. Start a local vLLM server

This starts an OpenAI-compatible vLLM server in the background. Logs are written to `vllm.log` so the notebook stays readable.

In [ ]:
%%bash
python -m vllm.entrypoints.openai.api_server \
  --model Qwen/Qwen3-VL-30B-A3B-Instruct \
  --host 127.0.0.1 \
  --port 8000 \
  --trust-remote-code \
  --max-model-len 120000 \
  2>&1 | tee vllm.log &

echo $! > vllm.pid
cat vllm.pid

## 2. Wait until vLLM is ready

The model can take a few minutes to load. This waits until the `/v1/models` endpoint responds.

In [ ]:
%%bash
until curl -s http://127.0.0.1:8000/v1/models >/dev/null; do
  sleep 5
done

echo "vLLM is ready"

## 3. Configure PaperScraper models

PaperScraper has separate text and vision profiles. Here both profiles point to the same local Qwen VL server.

In [ ]:
%%bash
export PAPERSCRAPER_MODEL_BASE_URL=http://127.0.0.1:8000/v1

ps_model_config text \
  --provider local \
  --model Qwen/Qwen3-VL-30B-A3B-Instruct \
  --base-url "$PAPERSCRAPER_MODEL_BASE_URL"

ps_model_config vision \
  --provider local \
  --model Qwen/Qwen3-VL-30B-A3B-Instruct \
  --base-url "$PAPERSCRAPER_MODEL_BASE_URL"

ps_model_status

## 4. Search for papers

This searches all configured search sources and writes a streamlined `papers.csv`. The `--count` option caps the number of results requested from each selected source.

In [ ]:
%%bash
ps_search "Lithium solid electrolyte" papers.csv \
  --source all \
  --count 10

## 5. Download available content

`ps_download` downloads text and PDFs using every configured source. Unpaywall uses `UNPAYWALL_EMAIL`, CORE uses `CORE_API_KEY`, and Elsevier uses `ELSEVIER_API_KEY`.

In [ ]:
%%bash
ps_download papers.csv papers \
  --format both \
  --source all

## 6. Scrape structured data

This runs the `sse` recipe over downloaded text and PDF images. `--image-context paper-text` gives the vision model the paper text as context while it analyzes images.

In [ ]:
%%bash
ps_scrape papers papers.csv sse \
  --mode text-images \
  --image-context paper-text

## 7. Stop vLLM

Run this when you are finished with the local model server.

In [ ]:
%%bash
if [ -f vllm.pid ]; then
  kill "$(cat vllm.pid)" 2>/dev/null || true
  rm vllm.pid
fi